# 07 — Surrogate Point Contributions (Observation-Level)

Fits a **Global surrogate** on each run's full prediction data, then computes
per-row feature contributions for a stratified sample.

| Model Type | Surrogate | Contribution formula |
|---|---|---|
| Regression | RE_Panel / GLS | `coef_j × x_j` (raw coefficient × raw feature value) |
| Classification | Decision Tree | SHAP Shapley values (positive-class) on surrogate DT |
| Classification | Logistic Regression (L1) | `coef_j × scaled_x_j` |

**Sampling**: 10% per Segment (stratified), minimum 20 rows per run.

**Outputs** (one file per run):  
`Outputs/surrogate_local/{model_key}_surrogate_{ds_key}.csv`

**No inputs to change.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [1]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.shap_utils      import load_model_artifacts
from src.surrogate_utils import compute_point_contributions

pd.set_option('display.max_columns', 50)

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
LOCAL_DIR    = os.path.join(OUTPUT_DIR, 'surrogate_local')
os.makedirs(LOCAL_DIR, exist_ok=True)

SAMPLE_FRAC = 0.10   # 10% per Segment for point-level contributions
MIN_ROWS    = 20     # floor for very small datasets

print('Libraries loaded')

Libraries loaded


## 1. Load Manifesto

In [2]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']

print(f'Runs scheduled : {len(dataset_configs)}')
print(f'Model types    : {sorted(set(r["model_type"] for r in dataset_configs))}')

Runs scheduled : 21
Model types    : ['Regression']


## 2. Surrogate Point Pipeline — All Runs

In [3]:
artifacts_cache = {}   # model_name -> features (only features needed here)

for run in dataset_configs:
    run_id     = run['run_id']
    model_name = run['model']
    model_type = run['model_type']

    print(f'\n[{run_id}]  type={model_type}', flush=True)

    # ── Load prediction file (from manifest path) ─────────────────────────────
    df = pd.read_csv(run['path'])

    # ── Load feature list from model pkl (cached per model) ──────────────────
    if model_name not in artifacts_cache:
        print(f'  Loading model artifacts...', flush=True)
        _, features, _ = load_model_artifacts(run)
        artifacts_cache[model_name] = features
        print(f'  Features : {len(features)}', flush=True)
    else:
        features = artifacts_cache[model_name]

    # ── Compute per-row surrogate contributions ───────────────────────────────
    try:
        out = compute_point_contributions(
            df, features, run,
            sample_frac=SAMPLE_FRAC,
            min_rows=MIN_ROWS,
        )
    except Exception as e:
        print(f'  [error] {e}', flush=True)
        continue

    # ── Save ─────────────────────────────────────────────────────────────────
    model_key = model_name.lower().replace(' ', '_')
    ds_key    = run['dataset'].lower()
    file_path = os.path.join(LOCAL_DIR, f'{model_key}_surrogate_{ds_key}.csv')
    out.to_csv(file_path, index=False)

    surr_cols = [c for c in out.columns if c.startswith('surr_')]
    print(f'  Saved: {os.path.basename(file_path)}'
          f'  ({len(out):,} rows  {len(surr_cols)} contribution columns)',
          flush=True)

print('\nSurrogate point pipeline complete.')


[FE_LSDV_DEV]  type=Regression


  Loading model artifacts...


  Features : 38


  Saved: fe_lsdv_surrogate_dev.csv  (3,309 rows  38 contribution columns)



[FE_LSDV_OOS]  type=Regression


  Saved: fe_lsdv_surrogate_oos.csv  (1,445 rows  38 contribution columns)


... (39 outputs left out)

  Saved: lstm_surrogate_oot_04.csv  (84 rows  30 contribution columns)



[LSTM_OOT_05]  type=Regression


  Saved: lstm_surrogate_oot_05.csv  (25 rows  30 contribution columns)



Surrogate point pipeline complete.


## 3. Quick Sample View

In [4]:
import glob
files = sorted(glob.glob(os.path.join(LOCAL_DIR, '*.csv')))
if files:
    preview = pd.read_csv(files[0])
    surr_cols = [c for c in preview.columns if c.startswith('surr_')]
    print(f'File : {os.path.basename(files[0])}')
    print(f'Rows : {len(preview):,}  |  Contribution columns : {len(surr_cols)}')
    print(f'Surrogate types : {sorted(set(c.split("_")[1] for c in surr_cols))}')
    display(preview.head(5))

File : fe_lsdv_surrogate_dev.csv
Rows : 3,309  |  Contribution columns : 38
Surrogate types : ['RE']


,Segment,Cust_ID,Year,WeekNum,UnitSales,pred_fe,surr_RE_Panel_global_t,surr_RE_Panel_week_sin,surr_RE_Panel_week_cos,surr_RE_Panel_quarter,surr_RE_Panel_is_year_start,surr_RE_Panel_is_year_end,surr_RE_Panel_lag_1,surr_RE_Panel_lag_2,surr_RE_Panel_lag_4,surr_RE_Panel_roll_mean_4,surr_RE_Panel_roll_mean_13,surr_RE_Panel_roll_std_4,surr_RE_Panel_wow_growth,surr_RE_Panel_momentum_4_13,surr_RE_Panel_is_promo,surr_RE_Panel_promo_intensity,surr_RE_Panel_pre_promo_1,surr_RE_Panel_pre_promo_2,surr_RE_Panel_post_promo_1,surr_RE_Panel_post_promo_2,surr_RE_Panel_weeks_since_promo,surr_RE_Panel_weeks_to_next_promo,surr_RE_Panel_promo_density_13,surr_RE_Panel_August,surr_RE_Panel_Christmas,surr_RE_Panel_Spring,surr_RE_Panel_NewYear,surr_RE_Panel_September,surr_RE_Panel_Thanksgiving,surr_RE_Panel_VetreansDay,surr_RE_Panel_any_holiday,surr_RE_Panel_promo_x_holiday,surr_RE_Panel_promo_x_windex,surr_RE_Panel_holiday_x_windex,surr_RE_Panel_Weekly_Index,surr_RE_Panel_cust_hist_mean,surr_RE_Panel_zero_streak,surr_RE_Panel_zero_frac_13
0,Bronze,CID00453,2025,16,5.0,5.01,31.413200,-3.073496,1.119125,0.0,-0.000000,0.0,0.000000,0.0,0.000000,-0.000000,0.0,-0.000000,0.0,-0.000000,0.0,0.000000,-0.0,-0.000000,0.0,0.000000,8.393842,-6.453245,-0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,-0.000000,0.0,0.0,0.000000,0.0,0.0,-7.872492,-3.766771
1,Bronze,CID00485,2024,16,727.0,4.57,17.800813,-3.073496,1.119125,0.0,-0.000000,0.0,0.004998,0.0,0.000000,-0.000543,0.0,-0.002775,0.0,-4.639981,0.0,0.000000,-0.0,-0.000000,0.0,0.000000,8.393842,-0.496403,-0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,-0.000000,0.0,0.0,0.000000,0.0,0.0,-0.000000,-3.013417
2,Bronze,CID00246,2025,4,6.0,4.46,28.271880,-1.527594,-2.794480,0.0,-118.483464,0.0,0.312372,0.0,0.000000,-0.033958,0.0,-0.173418,0.0,-3.711989,0.0,0.549326,-0.0,-0.000000,0.0,5.332564,0.000000,-0.000000,-3.059474,0.000000,0.0,0.0,186.192759,0.0,0.0,0.0,-62.276469,0.0,0.0,0.430730,0.0,0.0,-0.000000,-2.825079
3,Bronze,CID00270,2025,34,84.0,8.39,36.125180,2.705234,1.792800,0.0,-0.000000,0.0,0.417329,0.0,0.022984,-0.052295,0.0,-0.220846,0.0,-11.328358,0.0,0.000000,-0.0,-0.000000,0.0,0.000000,0.807100,-0.372303,-0.941377,57.082602,0.0,0.0,0.000000,0.0,0.0,0.0,-62.276469,0.0,0.0,0.627027,0.0,0.0,-0.000000,-2.318013
4,Bronze,CID00995,2023,18,0.0,2.91,4.711980,-2.705234,1.792800,0.0,-0.000000,0.0,0.000000,0.0,0.000000,-0.000000,0.0,-0.000000,0.0,-0.000000,0.0,0.000000,-0.0,-8.709106,0.0,0.000000,8.393842,-0.248202,-0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,-0.000000,0.0,0.0,0.000000,0.0,0.0,-1.312082,-3.766771
